## Geometry Clustering (train/dev)

`outputs/physics_feature_analysis_v2/geometry_sample_features.csv`의 기하학 피처를 사용해 구조물 형태를 군집화합니다.
- 목표: 소수 shape type으로 자동 분리
- 방법: 표준화 + KMeans + 실루엣 점수 기반 k 선택


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeClassifier, export_text

FEATURE_PATH = '../../outputs/physics_feature_analysis_v2/geometry_sample_features.csv'

df = pd.read_csv(FEATURE_PATH)
feature_cols = [c for c in df.columns if c not in ['split', 'sample_id', 'label']]

X = df[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True))

print('shape:', df.shape)
print('split counts:', df['split'].value_counts())
print('label counts:', df['label'].value_counts(dropna=False))


FileNotFoundError: [Errno 2] No such file or directory: '../../outputs/physics_feature_analysis_v2/geometry_sample_features.csv'

In [ ]:
# 1) k 탐색: 실루엣 점수
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

sil_rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster = km.fit_predict(Xs)
    sil = silhouette_score(Xs, cluster)
    sil_rows.append({'k': k, 'silhouette': sil})

sil_df = pd.DataFrame(sil_rows).sort_values('k')
print(sil_df.to_string(index=False))

best_k = int(sil_df.sort_values('silhouette', ascending=False).iloc[0]['k'])
print('
Best k by silhouette =', best_k)


In [ ]:
# 2) 최적 k로 군집화
k = best_k
kmeans = KMeans(n_clusters=k, random_state=42, n_init=50)
df['shape_cluster'] = kmeans.fit_predict(Xs)

print('cluster size:')
print(df['shape_cluster'].value_counts().sort_index())

print('
cluster x label (row-normalized):')
print(pd.crosstab(df['shape_cluster'], df['label'], normalize='index').round(3))

selected = [
    'front_slenderness',
    'front_width_frac',
    'front_base_width_frac',
    'front_top_width_frac',
    'top_area_frac',
    'top_support_width_frac',
    'front_tilt',
    'collapse_margin_proxy',
]
print('
cluster centers (mean):')
print(df.groupby('shape_cluster')[selected].mean().round(3))


### Cluster Visualization (balanced sampling)

클러스터 불균형 영향을 줄이기 위해, 각 클러스터에서 동일 개수 샘플을 뽑아 `front/top` 이미지를 시각화합니다.


In [ ]:
from pathlib import Path
import math
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 1) 군집 분포 확인
cluster_counts = df['shape_cluster'].value_counts().sort_index()
print(cluster_counts)

plt.figure(figsize=(8, 4))
plt.bar(cluster_counts.index.astype(str), cluster_counts.values)
plt.title('Cluster Size Distribution')
plt.xlabel('shape_cluster')
plt.ylabel('count')
plt.show()

# 2) split -> 디렉토리 매핑
def sample_dir(split: str, sample_id: str) -> Path:
    root = Path('..') / 'data'
    return root / split / sample_id

# 3) 각 클러스터에서 동일 개수 샘플링 후 시각화
n_per_cluster = 8
rng = random.Random(42)

clusters = sorted(df['shape_cluster'].unique())
for c in clusters:
    sub = df[df['shape_cluster'] == c]
    # 불균형 보정: 클러스터마다 동일 개수(가능한 만큼)
    n = min(n_per_cluster, len(sub))
    rows = sub.sample(n=n, random_state=42)

    print(f'\n[cluster {c}] size={len(sub)}, showing={n}')

    fig, axes = plt.subplots(nrows=n, ncols=2, figsize=(8, max(2.5*n, 4)))
    if n == 1:
        axes = [axes]

    for r, (_, row) in enumerate(rows.iterrows()):
        sid = row['sample_id']
        split = row['split']
        label = row.get('label', 'unknown')
        d = sample_dir(split, sid)

        front_p = d / 'front.png'
        top_p = d / 'top.png'

        ax_f = axes[r][0]
        ax_t = axes[r][1]

        if front_p.exists():
            ax_f.imshow(mpimg.imread(front_p))
        else:
            ax_f.text(0.5, 0.5, f'missing\n{front_p}', ha='center', va='center')
        ax_f.set_title(f'{sid} ({split}, {label}) - front')
        ax_f.axis('off')

        if top_p.exists():
            ax_t.imshow(mpimg.imread(top_p))
        else:
            ax_t.text(0.5, 0.5, f'missing\n{top_p}', ha='center', va='center')
        ax_t.set_title(f'{sid} ({split}, {label}) - top')
        ax_t.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# 3) 군집별 대표 샘플(중심과 가장 가까운 샘플)
centers = kmeans.cluster_centers_

for c in range(k):
    idx = np.where(df['shape_cluster'].values == c)[0]
    d2 = ((Xs[idx] - centers[c]) ** 2).sum(axis=1)
    nearest = idx[np.argsort(d2)[:5]]

    cols = [
        'sample_id', 'split', 'label',
        'front_slenderness', 'front_width_frac',
        'front_base_width_frac', 'front_top_width_frac',
        'top_support_width_frac', 'front_tilt'
    ]
    print(f'\n[cluster {c}] representatives')
    print(df.iloc[nearest][cols].to_string(index=False))


In [ ]:
# 4) 사람이 읽기 쉬운 규칙(해석용)
# 군집 ID를 얕은 의사결정나무로 근사하면, 형태 구분 기준선을 빠르게 볼 수 있음.
rule_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42)
rule_tree.fit(X, df['shape_cluster'])

print(export_text(rule_tree, feature_names=feature_cols))
print('approximation acc:', round(rule_tree.score(X, df['shape_cluster']), 4))
